<a href="https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/05-agents/01-agent-loop-from-scratch.zh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 从零实现 agent 循环

**目标：** 用原生 API 调用搭出一个能跑的 agent 循环，不用任何框架。

本项目是 [AI-EDF-NOTEBOOKS-CN](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN) 的一部分。


## 没接触过 agent？从这里开始

如果你从没搭过「AI agent」，别管那些炒作。这个想法很小。agent 就是一个**让模型调用你的函数的循环**：模型请求运行一个工具，你的代码运行它并把结果交还回去，如此重复直到任务完成。没有任何玄乎的东西，也不需要框架；你会在这个 notebook 里用约 40 行代码把整套东西搭出来。

本节分四步把 agent 搭起来，让你清楚自己身在何处：

| # | Notebook | 增加了什么 |
|---|---|---|
| **01** | **从零实现 agent 循环**（本篇） | 核心循环：模型 → 工具 → 重复 |
| 02 | 工具设计 | 让工具真正能被模型用好 |
| 03 | 护栏与预算 | 停止条件、成本上限，以及*什么时候一个普通 pipeline 胜过 agent* |
| 04 | MCP（概念） | 这个原生循环如何对应到行业工具标准 |

贯穿这四篇的一个理念：**自主 agent 是你最后才会去拿的工具，而不是最先拿的。** 它很强，但昂贵且不可预测，所以只有当任务真的无法事先脚本化时才用它。我们在这里搭这个循环是为了*理解*它；第 03 节教的是什么时候*不该*用。（前置条件：你已经在 `01-model-apis/02-tool-calling` 里见过工具调用；agent 就是它，放进了循环里。）

## 环境准备

每个 notebook 都是自包含的，所以接下来两个单元格会从零把它搭起来：

1. **安装依赖。** `aien` 包（就是本仓库）带有共享的环境准备助手，并会拉入 `groq` 客户端，这是本 notebook 唯一需要的依赖。
2. **加载你的 API key。** 到 [console.groq.com](https://console.groq.com/) 领一个免费 key（无需信用卡）。在 Colab 里，通过左侧边栏的 **钥匙图标** → **Add new secret** 添加它，名称必须严格写成 `GROQ_API_KEY`，粘贴值，然后打开 **Notebook access** 开关。在本地运行？把 `GROQ_API_KEY` 设为环境变量。

（完整走查和选模型指引见 [00-setup/00-environment.ipynb](https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/00-setup/00-environment.zh.ipynb)。）

In [ ]:
%pip install -q "git+https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN.git@main"

In [ ]:
from aien import setup

# 读取 GROQ_API_KEY（Colab Secrets 或本地环境变量），并返回一个就绪的
# Groq 客户端。传 model=... 可覆盖默认值；如果之后某次调用报 404，
# 就列出可用模型 —— 见 00-setup/00-environment.ipynb。
client, MODEL = setup()

## 为什么要有循环，以及 agent 不过是一个 while 循环

这是本 notebook 的任务：*「读取工作区里所有报销单，把它们加起来，写一个带总额的汇总文件。」* 单次模型调用做不到。模型不知道有哪些文件存在、读不了它们、做不了加法，也写不出结果。就算有工具（notebook 01-02 里的），*一次*调用也只能给你*一轮*工具使用。这个任务需要一条链：列出文件 → 逐个读取 → 计算 → 写入 → 汇报，其中每一步都取决于上一步返回了什么。

**agent** 就是跑这条链的东西。剥掉营销外壳，它只有四件事：

1. **一个模型**，它能请求工具调用。
2. **工具**：你用名字、描述和 schema 暴露出来的普通函数。
3. **一个循环**：调用模型；如果它请求了工具，就运行它们，把结果追加进去，再来一轮。
4. **停止条件**：模型不再请求工具，或者你撞上了回合上限。

就是这样。模型掌舵，循环不断把控制权交还给它，直到任务完成。整套东西放在约 40 行 Python 里，而亲手搭一遍是让你不再被 agent 框架搞晕的最快办法。我们这里就正好这么做，然后在上面那个多步任务上跑它。

> **动手之前先提醒一句，因为这是每个人都会犯的错。** 自主循环是解决问题*最贵、最不可预测*的方式：非确定性、逐步成本，以及 notebook 03 那一整套护栏机制。去拿它要**最后**拿，不是最先拿。如果你能事先把步骤写下来（抓取 → 抽取 → 总结 → 存储，每次运行都是同一形状），那就是一个 **pipeline**：把顺序写成代码，只在需要语言的步骤里调模型。*只有*当路径真的取决于更早步骤返回了什么时，才用循环。我们的任务符合这个条件，因为在列出文件之前你并不知道有哪些文件存在。我们在这里搭这个循环，因为它是你必须理解的原子；notebook 03 会把「什么时候 pipeline 胜过 agent」讲得严谨。整个过程都要记住这个偏向更简单工具的倾向。

## 工具

我们需要能在 Colab 里安全运行、不依赖外部服务的工具。两个就够了：

- 一个极小的**内存文件系统**（就是一个 dict），带 `list_files`、`read_file` 和 `write_file`。
- 一个**计算器**，用来求算术表达式。

每个工具都有两半：*实现*（一个 Python 函数）和*定义*（名字 + 描述 + JSON schema），后者是模型看到的。模型永远看不到你的代码，只看得到定义。记住这一点；它会成为 notebook 02 的整个主题。

In [ ]:
import json
import re

# 一个假文件系统：path -> content。安全、可检视、可重置。
FS = {
    "notes/monday.md": "Team lunch: $42.50\nTaxi to client site: $18.00\n",
    "notes/tuesday.md": "Conference tickets, 2 x $150.00 each\n",
    "notes/wednesday.md": "Cloud credits top-up: $75.25\nCoffee for the workshop: $23.10\n",
}

def list_files():
    return "\n".join(sorted(FS)) if FS else "(no files)"

def read_file(path):
    if path not in FS:
        return f"Error: no file at '{path}'. Call list_files to see what exists."
    return FS[path]

def write_file(path, content):
    FS[path] = content
    return f"Wrote {len(content)} characters to {path}."

def calculator(expression):
    # 只有数字、运算符、括号、点、空格 —— 没有名字，没有属性访问。
    if not re.fullmatch(r"[0-9+\-*/(). ]+", expression):
        return "Error: only numbers and + - * / ( ) are allowed."
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"

IMPLS = {
    "list_files": list_files,
    "read_file": read_file,
    "write_file": write_file,
    "calculator": calculator,
}

def execute_tool(name, args):
    fn = IMPLS.get(name)
    if fn is None:
        return f"Error: unknown tool '{name}'."
    try:
        return str(fn(**args))
    except TypeError as e:
        return f"Error: bad arguments for {name}: {e}"

# 按 Groq 使用的 OpenAI 兼容格式写工具定义：
# 每个都是 {"type": "function", "function": {name, description, parameters}}。
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "List every file path in the workspace, one per line. "
                           "Use this first if you don't know which files exist.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read the full contents of one file. Returns the text, "
                           "or an error if the path doesn't exist.",
            "parameters": {
                "type": "object",
                "properties": {"path": {"type": "string", "description": "Exact path from list_files."}},
                "required": ["path"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": "Create or overwrite a file with the given content. "
                           "Returns a confirmation.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string"},
                    "content": {"type": "string"},
                },
                "required": ["path", "content"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate an arithmetic expression using + - * / and parentheses, "
                           "e.g. '42.50 + 18.00'. Use this for all math instead of computing "
                           "in your head. Returns the numeric result.",
            "parameters": {
                "type": "object",
                "properties": {"expression": {"type": "string"}},
                "required": ["expression"],
            },
        },
    },
]


## 循环

API 是无状态的：每个请求都携带到目前为止的完整对话，而 `messages` 列表*就是* agent 的全部状态。每一轮：

1. 把 `messages` 发给模型。
2. 如果 `finish_reason == "tool_calls"`，模型的回复里含一个或多个工具调用。原样追加这条回复，执行每一个被请求的工具，并把每个结果作为各自的 `role: "tool"` 消息发回去（靠 `tool_call_id` 配对）。
3. 否则模型就说完了，返回它的文本。

`max_turns` 这个上限不是可有可无的装饰。每一轮都是要付费的 API 调用，所以一个没有上限的循环就是一笔没有上限的账单。本仓库里每个循环都带一个。

我们还会打印每一轮的 trace。trace 就是教学法：它是你看见模型如何规划的方式。

In [ ]:
def run_agent(task, tools, max_turns=10):
    """一个 agent：一个模型、一些工具、一个 while 循环，加上停止条件。"""
    messages = [{"role": "user", "content": task}]

    for turn in range(1, max_turns + 1):
        response = client.chat.completions.create(
            model=MODEL,
            max_tokens=1024,
            tools=tools,
            messages=messages,
        )
        choice = response.choices[0]
        msg = choice.message

        # trace：模型这一轮产出的任何文本（它「把思考说出口」）
        if msg.content and msg.content.strip():
            print(f"[turn {turn}] model: {msg.content.strip()[:200]}")

        # 停止条件 #1：模型没有请求工具 —— 它说完了。
        if choice.finish_reason != "tool_calls":
            return msg.content or ""

        # 原样追加 assistant 这一轮（它携带 tool_calls 列表）。
        messages.append(msg)

        # 执行每一个被请求的工具；把每个结果作为各自的 tool 消息追加。
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            result = execute_tool(tc.function.name, args)
            print(f"[turn {turn}] tool : {tc.function.name}({tc.function.arguments[:120]})")
            print(f"[turn {turn}]   -> {result.strip()[:120]}")
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result,
            })

    # 停止条件 #2：回合上限。大声失败，而不是悄悄失败。
    return f"Stopped: hit max_turns={max_turns} before the task finished."


这就是整个 agent。两个停止条件，一份 trace，没有框架。

## 跑它

下面这个任务没法一次做完：模型必须自己发现文件、读它们、做算术、写出结果，这是一条 3–5 次工具调用的链，后面的步骤依赖前面的结果。

In [ ]:
result = run_agent(
    "Read all the notes files in the workspace, add up every expense using the "
    "calculator, and write a file called summary.md that states the total. "
    "Then tell me the total.",
    tools=TOOLS,
)
print("\n=== final answer ===")
print(result)


In [ ]:
# 副作用是真的：agent 写进了我们的假文件系统。
print(FS.get("summary.md", "(summary.md was not written)"))


## trace 展示了什么

跑一遍上面的单元格，从上到下读一遍 trace。有几件事值得注意：

- **模型在无人指示的情况下跨回合规划。** 它通常先列文件、再读、再用计算器算，然后写。没有任何人把顺序脚本化；它是从循环加上还不错的工具描述里自然涌现出来的。
- **周二的记录需要真正的推理**：「2 x $150.00 每份」不是一个能照抄的数字。看看它发给计算器的表达式是怎么处理这个的。
- **`messages` 列表就是 agent 的全部状态。** 没有隐藏记忆。如果你最后把 `messages` 打印出来，你会看到完整的运行过程：任务、工具调用、结果、答案。
- **每一轮都会重发完整历史。** 第 5 轮的请求一字不差地包含第 1–4 轮。上下文增长就是成本增长（长运行下大致是二次方增长），这就是 notebook 03 要给这个循环上预算的原因，也是工具*输出体积*在 notebook 02 里如此重要的原因。

跑两遍，trace 的细节会有差异（读取顺序、精确的表达式），但都会收敛到同一个答案。这种非确定性是正常的；这也是为什么生产环境里的 agent 需要我们后面加的护栏。

## 框架加了什么

LangGraph、CrewAI、OpenAI Agents SDK、Claude 的 tool runner：它们的核心都在跑这同一个循环。它们在上面加的是：

- **持久化** —— 给 `messages` 做检查点，让一次运行能挺过进程重启。
- **可观测性** —— 结构化的 trace，取代我们的 `print` 语句。
- **交接 / 多 agent 接线** —— 在几个使用不同工具的循环之间路由。
- **重试、速率限制、人工审批钩子** —— 运维管路。

全都有用；没有一个是魔法。亲手搭过这个循环之后，你现在就能正好按这些维度评估一个框架：它持久化什么、给我看什么、把什么接在一起，以及它隐藏了什么？



## 练习

1. **加一个 `delete_file` 工具**（实现、定义和分发入口），然后给 agent 一个需要它的任务，例如「把所有笔记合并成一个文件，并删除原文件」。看 trace 确认它选的顺序。
2. **故意弄坏一个描述。** 把 `calculator` 的描述改成只有「做数学。」并去掉那个例子。把同一个任务重跑三次。模型还会用它吗，还是开始在心里做算术？（这就是 notebook 02 的论点，缩小版。）
3. **打印 token 成本曲线。** 每一轮捕获 `response.usage.prompt_tokens` 并打印。确认即使你的任务文本从未改变，输入 token 每一轮都在增长，然后按当前单价估算一次 50 轮运行的成本。
4. **返回 trace，而不只是答案。** 让 `run_agent` 额外返回一个 `(turn, tool_name, args, result)` 元组列表，并写五行代码回答：共几轮、几次工具调用、哪个工具被调用得最多？